[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/03_autogen.ipynb)

# Part 3 — Conversational multi-agent (AutoGen)

So far we have considered one model interacting with one system prompt; either proceeding according to its own wishes (Part 1: ReAct) or following a rigid execution graph (Part 2: LangGraph). Here we will consider a **conversational multi-agent** system which instead runs multiple agents. Each one will be given its own "personality" or goals, and we will let them interact with each other until they reach some notion of consensus. At the end, a transcript describing their interactions can provide important information about how a "good" decision was reached.

AutoGen requires us to specify three pieces:

- instantiations of **`AssistantAgent`** — a model plus a system prompt and a name. This is where we specify the different personalities, the job they're supposed to do, and how they interact;
- a **team** — the object that decides the order in which the `AssistantAgent`s interact. `RoundRobinGroupChat` simply cycles through the agents in order;
- a **termination condition** — a rule that decides when the conversation has finished.

Nobody writes the order of argument down. Each agent sees the transcript so far and responds to it, so what gets discussed depends on what the previous speaker said.

**What does this notebook do?** We set up three agents with deliberately opposed instructions — one proposes a discretization, one attacks it, one judges whether the attack was answered — and let them argue about a real numerical question until the judge approves or we hit a message cap.

**You are done when** the transcript ends either with the critic saying APPROVE, or after ten messages with the disagreement still open.

## 3.0 API Setup

Same key as Part 0, but a different client. AutoGen reaches Gemini through Gemini's OpenAI-compatible endpoint, so we build an `OpenAIChatCompletionClient` and point it at a `gemini-` model. The comments in the cell explain the three arguments that are not obvious.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0" autogen-agentchat "autogen-ext[openai]"

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

MODEL = "gemini-3.1-flash-lite"

# AutoGen talks to Gemini through Gemini's OpenAI-compatible endpoint. Any model name
# starting with "gemini-" gets base_url set automatically.
#
# model_info must be passed explicitly: AutoGen's built-in registry stops at gemini-2.5,
# and its fallback for unrecognised models sets function_calling=False -- which would
# silently disable tool use rather than raise.
model_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=API_KEY,
    model_info=ModelInfo(vision=False, function_calling=True, json_output=True,
                         family="unknown", structured_output=True),
    # gemini-3.x thinks by default and thinking cannot be disabled on 3-series models,
    # only turned down. Without this each turn takes ~30s. "minimal" is faster still.
    reasoning_effort="low",
    # The openai client defaults to a 600s read timeout and 2 retries, so a rejected
    # or rate-limited request looks like a hang for many minutes. Fail fast instead.
    timeout=60.0,
    max_retries=1,
)


## 3.1 The syntax

Each agent is an `AssistantAgent` carrying a name, a model client, and a system prompt. The prompts are where the work happens: the three below are written to disagree, not to cooperate.

`RoundRobinGroupChat` takes the list of agents and cycles through them, appending each reply to a shared transcript that all of them can see.

The run needs a rule for when to stop, and we give it two:

- `TextMentionTermination("APPROVE")` fires when the word APPROVE appears in any message.
- `MaxMessageTermination(10)` fires once ten chat messages exist. With three agents that is the task plus nine replies, or three full rounds.

The two are joined with `|`. AutoGen overloads that operator to mean *or*, so the combined rule fires as soon as either one does — the critic approves, or we hit ten messages, whichever comes first. The message cap is there so a critic that never approves cannot quietly spend your daily quota.

In [ ]:
modeler = AssistantAgent(
    "modeler", model_client=model_client,
    system_message="You propose discretizations for PDE problems. Be concrete and specific: "
                   "name the method and the parameters. Two or three sentences. Respond to "
                   "objections rather than repeating yourself.")

analyst = AssistantAgent(
    "analyst", model_client=model_client,
    system_message="You are a numerical analyst. Challenge the modeler's proposal on "
                   "stability, conditioning, and convergence rate. Be specific about the "
                   "failure mode you are worried about. Do not be agreeable -- your value to "
                   "this conversation is the objection. Two or three sentences.")

critic = AssistantAgent(
    "critic", model_client=model_client,
    system_message="You judge the exchange between a modeler and a numerical analyst. If the "
                   "analyst's objection has been genuinely answered, reply with exactly "
                   "APPROVE followed by one sentence of justification. Otherwise state in one "
                   "sentence what is still unresolved. Do not say APPROVE merely because the "
                   "discussion is polite or has gone on a while.")

# APPROVE ends it; the message cap is the backstop so a stubborn critic cannot burn the
# free-tier daily budget.
team = RoundRobinGroupChat(
    [modeler, analyst, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(10),
)

## 3.2 A question worth arguing about

A conversation only produces something a single agent would not if the question has real tension in it. Arithmetic has none: there is one right answer to `13 * 47 + 8` and nothing to debate.

Steady advection-diffusion at Péclet number ~500 does have tension. Standard Galerkin discretization oscillates badly in that regime, and every fix — upwinding, SUPG, smaller elements — costs something in accuracy, cost, or complexity. Reasonable people disagree, which is what gives the analyst something to push on.

In [ ]:
TASK = ("Choose a spatial discretization for steady advection-diffusion at Peclet number "
        "~500 on an unstructured triangular mesh. State the method and any stabilization "
        "parameter.")

await Console(team.run_stream(task=TASK))

The client holds an open HTTP session. Close it when you are finished so the notebook does not leave a connection dangling.

In [ ]:
await model_client.close()

## 3.3 When to reach for conversation

Reach for a conversation when the argument itself is the output — when you want the objections recorded, not just the conclusion.

Three cautions, because this pattern is the easiest one to fool yourself with:

- **Agreement between agents is weak evidence.** All three share a training corpus and a tendency toward agreeableness. A critic that approves everything is measuring nothing.
- **Give roles different tools, not just different adjectives.** An analyst that can run a stability calculation contributes something an analyst told to "be skeptical" cannot.
- **Read the transcript for early approval.** If the critic approves before the objection was answered, you have seen this pattern's characteristic failure.

The smallest version worth building is two agents where one has a tool the other lacks.

## 3.4 async/await, and running this code outside Colab

To understand the code above, you'll need to understand async/await and how they are used to perform nonblocking code execution. Each agent's turn is spent waiting on an HTTP response from the model API, and `run_stream` hands back messages as they arrive rather than after the conversation finishes — which is what lets us print the exchange as it happens.

AutoGen's API is asynchronous. `team.run_stream`, `Console` and `model_client.close` are declared `async def`, so calling one returns a coroutine object rather than running it; `await` runs it and waits for the result. Colab and Jupyter execute each cell inside an event loop, which is why `await` works at the top level of a cell here. In a `.py` script it is a `SyntaxError`, and the equivalent is:

```python
import asyncio

async def main():
    await Console(team.run_stream(task=TASK))
    await model_client.close()

asyncio.run(main())
```

For an introduction to `asyncio`, see [Coroutines and Tasks](https://docs.python.org/3/library/asyncio-task.html) in the Python documentation.

## 3.5 Further reading

- **[AutoGen documentation](https://microsoft.github.io/autogen/stable/)** — start here.
- **[AgentChat user guide](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/index.html)** — the layer this notebook uses.
- **[Teams](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/teams.html)** and **[termination conditions](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/termination.html)** — the two pieces we configured.
- **[SelectorGroupChat](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/selector-group-chat.html)** — picks the next speaker with a model instead of cycling, which is the usual next step after round-robin.
- **[Gemini's OpenAI compatibility layer](https://ai.google.dev/gemini-api/docs/openai)** — why the client in 3.0 is an OpenAI one, and what `reasoning_effort` does.